# SecurePay Vision - Exploratory Data Analysis

**Final Project: AI-Powered Fraud Detection for UMKM**

Notebook ini berisi Exploratory Data Analysis (EDA) untuk dataset transaksi digital UMKM.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
COLORS = {'normal': '#00ffcc', 'fraud': '#ef4444', 'blue': '#00d4ff', 'purple': '#7c3aed'}

print('Libraries loaded!')

In [ ]:
df = pd.read_csv('../dataset/transactions.csv')
print('Dataset Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
df.head()

In [ ]:
print('=== BASIC STATISTICS ===')
print(df.describe())
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print('\n=== CLASS DISTRIBUTION ===')
print(df['is_fraud'].value_counts())
print(f'Fraud Rate: {df["is_fraud"].mean():.1%}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('SecurePay Vision - Transaction Analysis', fontsize=16, color='white')

# 1. Class distribution
axes[0,0].bar(['Normal', 'Fraud'], df['is_fraud'].value_counts().values,
               color=[COLORS['normal'], COLORS['fraud']])
axes[0,0].set_title('Class Distribution')

# 2. Amount distribution by class
normal_amounts = df[df['is_fraud']==0]['amount']
fraud_amounts = df[df['is_fraud']==1]['amount']
axes[0,1].hist(np.log1p(normal_amounts), bins=30, alpha=0.7, color=COLORS['normal'], label='Normal')
axes[0,1].hist(np.log1p(fraud_amounts), bins=30, alpha=0.7, color=COLORS['fraud'], label='Fraud')
axes[0,1].set_title('Log Amount Distribution')
axes[0,1].legend()

# 3. Transaction by hour
hour_normal = df[df['is_fraud']==0].groupby('hour').size()
hour_fraud = df[df['is_fraud']==1].groupby('hour').size()
axes[0,2].plot(hour_normal.index, hour_normal.values, color=COLORS['normal'], label='Normal', marker='o')
axes[0,2].plot(hour_fraud.index, hour_fraud.values, color=COLORS['fraud'], label='Fraud', marker='x')
axes[0,2].set_title('Transactions by Hour')
axes[0,2].legend()

# 4. Payment method distribution
payment_counts = df['payment_method'].value_counts().head(8)
axes[1,0].barh(payment_counts.index, payment_counts.values, color=COLORS['blue'])
axes[1,0].set_title('Top Payment Methods')

# 5. Fraud by fraud type
fraud_types = df[df['is_fraud']==1]['fraud_type'].value_counts()
axes[1,1].pie(fraud_types.values, labels=fraud_types.index, autopct='%1.1f%%',
               colors=plt.cm.Set2.colors)
axes[1,1].set_title('Fraud Type Distribution')

# 6. Amount boxplot by class
df_plot = df.copy()
df_plot['log_amount'] = np.log1p(df_plot['amount'])
df_plot['Class'] = df_plot['is_fraud'].map({0: 'Normal', 1: 'Fraud'})
df_plot.boxplot(column='log_amount', by='Class', ax=axes[1,2])
axes[1,2].set_title('Amount by Class (Log Scale)')

plt.tight_layout()
plt.savefig('../reports/eda_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('EDA plots saved!')

In [ ]:
# Feature Correlation Heatmap
numeric_cols = ['amount', 'amount_log', 'hour', 'day_of_week', 
                'amount_zscore', 'is_odd_hours', 'is_weekend', 'is_fraud']

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', color='white', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/correlation_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Correlation heatmap saved!')